# 02 — Tokenizer and materialization, gen2

The second notebook of the gen2 set: the DSL `Tokenizer` pipeline, and the
single materialization morphism (`hllset-morphisms` — LUT-first, TF only
for ambiguity). The legacy strategy zoo (De Bruijn, ngram cross-validate,
homogeneous consensus, pluggable engines) is discharged: those live in the
frozen line and will return from the algebra when needed.


In [2]:
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-dsl" }
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-core" }
:dep hllset-lut = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-lut" }
:dep hllset-morphisms = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-morphisms" }


In [3]:
use hllset_dsl::{LatticeElement, Tokenizer};
use hllset_core::HLLSet;
use hllset_contracts::{token_in_bytes, token_in_bytes_le};
use hllset_lut::LutIndex;
use hllset_morphisms::{materialize, Ingest};
println!("gen2 tokenizer + morphisms loaded");


gen2 tokenizer + morphisms loaded


---
## Tokenizer

Bytes → [Pattern Match] → Tokens → [Normalize] → [N-grams] → [Boundary Pad] → `LatticeElement`.


In [4]:
let tok = Tokenizer::new();
let tokens = tok.extract(b"The cat sat on the mat");
for (i, t) in tokens.iter().enumerate() {
    println!("token {}: {:?}", i + 1, std::str::from_utf8(t).unwrap());
}


token 1: "The"
token 2: "cat"
token 3: "sat"
token 4: "on"
token 5: "the"
token 6: "mat"


()

In [5]:
let tok = Tokenizer::new().lowercase();
let tokens = tok.tokenize(b"The CAT sat ON the MAT");
for t in &tokens {
    print!("{} ", std::str::from_utf8(t).unwrap());
}
println!();


the cat sat on the mat 


In [6]:
let tok = Tokenizer::new().lowercase();
let elem: LatticeElement = tok.apply(b"Hello World Lua DSL");
println!("key:          {}", elem.key());
println!("cardinality:  {:.1}", elem.cardinality());
println!("popcount:     {}", elem.popcount());
let raw: HLLSet = elem.into_hllset();
println!("raw popcount: {}", raw.popcount());


key:          h:26a63fa47b1df1e3ba06a8116ace7d50dc1fd1b3
cardinality:  4.0
popcount:     4
raw popcount: 4


In [7]:
let tok = Tokenizer::new().lowercase().ngrams(1, 3);
let tokens = tok.tokenize(b"the cat sat on the mat");
for t in &tokens {
    let s = String::from_utf8_lossy(t).replace("\0", "|");
    print!("{} ", s);
}
println!();


the cat sat on the mat the|cat cat|sat sat|on on|the the|mat the|cat|sat cat|sat|on sat|on|the on|the|mat 


---
## Materialization: one morphism, LUT-first

Ingest is complete and single-touch; materialization gathers candidates from
the pointed LUTs and consults TF only for collided bits.


In [8]:
let tok = Tokenizer::new().lowercase();
let text = b"lattice algebra bss morphism tokenization materialization";
let toks = tok.tokenize(text);
let mut ingest: Ingest = Ingest::new();
ingest.ingest_tokens(toks.iter().map(|t| t.as_slice()));
println!("ingested {} tokens in one touch; seed-0 popcount {}", ingest.touched, ingest.hllset(0).popcount());


ingested 6 tokens in one touch; seed-0 popcount 6


In [9]:
let tok = Tokenizer::new().lowercase();
let text = b"lattice algebra bss morphism tokenization materialization";
let toks = tok.tokenize(text);
let mut ingest: Ingest = Ingest::new();
ingest.ingest_tokens(toks.iter().map(|t| t.as_slice()));
let sketch: HLLSet = ingest.hllset(0).clone();
let restored = materialize(&[(&sketch, ingest.lut(0))], ingest.tf());
let names: Vec<String> = restored.iter().map(|t| String::from_utf8_lossy(t).to_string()).collect();
println!("restored {} tokens (lossy where bits collide): {:?}", restored.len(), names);


restored 6 tokens (lossy where bits collide): ["algebra", "bss", "lattice", "materialization", "morphism", "tokenization"]


In [10]:
// Gate vs Slice are the same morphism M(H, L) at different LUT nodes.
let tok = Tokenizer::new().lowercase();
let toks = tok.tokenize(b"alpha beta gamma delta epsilon");
let vocab = vec![toks[0].clone(), toks[2].clone()]; // gate node: subset
let mut ingest: Ingest = Ingest::new();
ingest.ingest_tokens(toks.iter().map(|t| t.as_slice()));
let sketch: HLLSet = ingest.hllset(0).clone();
let mut slice_lut: LutIndex = LutIndex::default();
for t in &toks { slice_lut.insert_token(t.clone()); }
let mut gate_lut: LutIndex = LutIndex::default();
for t in &vocab { gate_lut.insert_token(t.clone()); }
let slice = materialize(&[(&sketch, &slice_lut)], ingest.tf());
let gate = materialize(&[(&sketch, &gate_lut)], ingest.tf());
println!("slice (full LUT) restores {} candidates; gate (vocab node) restores {}", slice.len(), gate.len());


slice (full LUT) restores 5 candidates; gate (vocab node) restores 2


In [11]:
// TF resolves bit ambiguity — and only ambiguity (notebook-08 collision pin).
let mut ci: Ingest = Ingest::new();
let c1 = token_in_bytes_le(262).to_vec();
let c2 = token_in_bytes_le(48_300).to_vec();
ci.ingest_token(&c1);
ci.ingest_token(&c1); // TF(c1) = 2
ci.ingest_token(&c2); // TF(c2) = 1
let mut coll: HLLSet = HLLSet::new();
coll.add_bit(759 * 32 + 0);
let restored = materialize(&[(&coll, ci.lut(0))], ci.tf());
println!("collided bit restores {:?} — TF-max wins", restored);
println!("winner is tid262LE: {}", restored == std::collections::BTreeSet::from([c1.clone()]));


collided bit restores {[6, 1, 0, 0]} — TF-max wins
winner is tid262LE: true


---
## Summary

Tokenizer pipelines words and n-grams into `LatticeElement`s; the two
morphisms (`Ingest`, `materialize`) move between tokens and sketches with
one LUT lattice and one TF table. Discharged from the frozen line: De
Bruijn reconstruction, ngram cross-validation, homogeneous 2-of-3
consensus, and the pluggable engine registry — they will be re-expressed
from the algebra when they return.
